# Phase 3 — Encoder Fine-Tuning Ablation + Leakage Demonstration

Ömer Faruk Merey — Middle East Technical University

Phase 2 found that with **frozen** CLIP encoders the cross-attention architecture drives most of the gain and clean captions help only conditionally. Phase 3 introduces the *severe change* — unfreezing the encoder via **LoRA** — and a **leakage demonstration** (`Rleak`, raw hybrid captions with GT percentages).

| Group | Conditions | Encoder | Notes |
|---|---|---|---|
| Frozen (Phase 2) | R0a, R0b, R2b | frozen | loaded from `reports/phase-2/phase2_results.csv` |
| LoRA (Phase 3) | R0a-lora, R0b-lora, R2b-lora | LoRA r=8 | online training, 3 seeds |
| Leakage (Phase 3) | Rleak | frozen | raw `hybrid_gemma3-4b`, precomputed |


## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2
%pip install -q git+https://github.com/openai/CLIP.git spacy wandb tqdm scipy

In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parents[1]
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import torch
import clip
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.dataset import CONDITIONS, COMPOSITION_CLASSES
from src.precompute import precompute_images, precompute_texts
from src.train import TrainConfig, train_one_condition
from src.eval import predict_split, compute_metrics, aggregate_runs

DATASET_DIR = REPO / 'dataset'
IMAGES_DIR = DATASET_DIR / 'images'
CAPTIONS_CSV = DATASET_DIR / 'captions.csv'
CKPT_DIR = REPO / 'checkpoints'; CKPT_DIR.mkdir(exist_ok=True)
PRECOMPUTED_DIR = REPO / 'precomputed'; PRECOMPUTED_DIR.mkdir(exist_ok=True)
REPORTS_DIR = REPO / 'reports' / 'phase-3'; REPORTS_DIR.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('device:', device)

clip_model, preprocess = clip.load('ViT-B/32', device=device)
clip_model.eval()
tokenizer = clip.tokenize

SEEDS = [42, 1337, 2024]

## 2. Leakage demonstration (Rleak)

Train on raw `hybrid_gemma3-4b` captions (GT percentages intact). Frozen CLIP, so the precompute pipeline applies. Expected: test MSE collapses far below the leakage-free R0a (0.00707), confirming the model reads the answer from the text rather than the image.

In [ ]:
# Precompute the raw-hybrid text embeddings (images.pt already exists from Phase 2)
precompute_images(CAPTIONS_CSV, IMAGES_DIR, PRECOMPUTED_DIR, clip_model, preprocess, device)
precompute_texts(CAPTIONS_CSV, PRECOMPUTED_DIR, clip_model, tokenizer, device,
                 conditions=('Rleak',))
print(sorted(p.name for p in PRECOMPUTED_DIR.iterdir()))

In [ ]:
USE_WANDB = True
for seed in SEEDS:
    ckpt = CKPT_DIR / f'Rleak_seed{seed}_best.pt'
    if ckpt.exists():
        print(f'-- skipping Rleak seed={seed}'); continue
    cfg = TrainConfig(condition='Rleak', seed=seed, epochs=30, batch_size=128,
                      use_wandb=USE_WANDB, checkpoint_dir=str(CKPT_DIR),
                      precomputed_dir=str(PRECOMPUTED_DIR),
                      wandb_tags=['phase3', 'leakage'])
    print(f'\n=== train Rleak seed={seed} ===')
    train_one_condition(cfg, CAPTIONS_CSV, IMAGES_DIR, clip_model, preprocess, tokenizer, device)

## 3. LoRA fine-tuning (R0a / R0b / R2b)

Online training (precompute is invalid once the encoder learns). On M4 each run is ~30–60 min; resume-friendly via checkpoints. `LORA_EPOCHS` defaults to 15 since fine-tuning converges faster than the frozen head and online training is expensive — raise to 30 on a CUDA GPU.

In [ ]:
LORA_CONDITIONS = ['R0a', 'R0b', 'R2b']
LORA_EPOCHS = 15

for cond in LORA_CONDITIONS:
    for seed in SEEDS:
        ckpt = CKPT_DIR / f'{cond}-lora_seed{seed}_best.pt'
        if ckpt.exists():
            print(f'-- skipping {cond}-lora seed={seed}'); continue
        cfg = TrainConfig(
            condition=cond, seed=seed, epochs=LORA_EPOCHS, batch_size=32,
            lr=5e-5, num_workers=2, use_wandb=USE_WANDB,
            checkpoint_dir=str(CKPT_DIR),
            lora=True, lora_r=8, lora_alpha=16, lora_last_k=6,
            wandb_tags=['phase3', 'lora'],
        )
        print(f'\n=== train {cond}-lora seed={seed} ===')
        train_one_condition(cfg, CAPTIONS_CSV, IMAGES_DIR, clip_model, preprocess, tokenizer, device)

## 4. Evaluate Phase 3 checkpoints on the test set

In [ ]:
rows = []

# Leakage (frozen, precomputed)
for seed in SEEDS:
    ckpt = CKPT_DIR / f'Rleak_seed{seed}_best.pt'
    if not ckpt.exists():
        print(f'!! missing {ckpt}'); continue
    out = predict_split('Rleak', seed, CAPTIONS_CSV, IMAGES_DIR, clip_model, preprocess,
                        tokenizer, device, ckpt, split='test',
                        precomputed_dir=str(PRECOMPUTED_DIR), batch_size=128)
    m = compute_metrics(out['pred'], out['gt']); m['condition'] = 'Rleak'; m['seed'] = seed
    rows.append(m)

# LoRA (online + adapters)
for cond in LORA_CONDITIONS:
    for seed in SEEDS:
        ckpt = CKPT_DIR / f'{cond}-lora_seed{seed}_best.pt'
        if not ckpt.exists():
            print(f'!! missing {ckpt}'); continue
        out = predict_split(cond, seed, CAPTIONS_CSV, IMAGES_DIR, clip_model, preprocess,
                            tokenizer, device, ckpt, split='test', batch_size=64,
                            lora=True, lora_r=8, lora_alpha=16, lora_last_k=6)
        m = compute_metrics(out['pred'], out['gt'])
        m['condition'] = f'{cond}-lora'; m['seed'] = seed
        rows.append(m)

phase3_df = pd.DataFrame(rows)
phase3_df.to_csv(REPORTS_DIR / 'phase3_results.csv', index=False)
phase3_df.groupby('condition')[['mse','mae','rmse','kl']].agg(['mean','std'])

## 5. Frozen vs LoRA comparison

In [ ]:
# Load frozen Phase 2 numbers
frozen = pd.read_csv(REPO / 'reports' / 'phase-2' / 'phase2_results.csv')
frozen_mse = frozen.groupby('condition')['mse'].mean()
p3_mse = phase3_df.groupby('condition')['mse'].mean()

comp = pd.DataFrame({
    'frozen': [frozen_mse.get(c, np.nan) for c in ['R0a','R0b','R2b']],
    'lora':   [p3_mse.get(f'{c}-lora', np.nan) for c in ['R0a','R0b','R2b']],
}, index=['R0a','R0b','R2b'])
comp['delta'] = comp['lora'] - comp['frozen']
comp['rel_%'] = 100 * comp['delta'] / comp['frozen']
print(comp)

x = np.arange(len(comp)); w = 0.38
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(x-w/2, comp['frozen'], w, label='frozen (Phase 2)', color='lightgray', hatch='//', edgecolor='black')
ax.bar(x+w/2, comp['lora'],   w, label='LoRA (Phase 3)',   color='white',     hatch='xx', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(comp.index)
ax.set_ylabel('Test MSE'); ax.set_title('Frozen vs LoRA encoder fine-tuning')
ax.legend()
plt.tight_layout(); plt.savefig(REPORTS_DIR / 'phase3_frozen_vs_lora.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. Leakage demonstration bar

`Rleak` (raw hybrid, GT percentages in text) vs the leakage-free frozen R0a and best LoRA condition. A near-zero `Rleak` MSE confirms the Phase 1 leakage hypothesis: the model recovers the targets from the caption text.

In [ ]:
rleak_mse = phase3_df.query('condition == "Rleak"')['mse'].mean()
bars = {
    'R0a (frozen,\nleakage-free)': frozen_mse.get('R0a', np.nan),
    'best LoRA': float(np.nanmin(comp['lora'])),
    'Rleak (raw\nhybrid, leaky)': rleak_mse,
}
fig, ax = plt.subplots(figsize=(6,4))
b = ax.bar(list(bars.keys()), list(bars.values()),
           color='lightgray', edgecolor='black')
for bar, h in zip(b, ['', '//', 'xx']):
    bar.set_hatch(h)
ax.set_ylabel('Test MSE'); ax.set_title('Leakage control vs leakage-free conditions')
for bar, v in zip(b, bars.values()):
    ax.text(bar.get_x()+bar.get_width()/2, v, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.savefig(REPORTS_DIR / 'phase3_leakage.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Rleak MSE = {rleak_mse:.5f}  vs  frozen R0a = {frozen_mse.get("R0a"):.5f}')